In [ ]:
import os, sys, glob, json, time, subprocess, shutil
T0 = time.time(); TRAN_GIO = 11.0
subprocess.run("pip uninstall -y -q torchao", shell=True)
WS = "/kaggle/working"; OUT = f"{WS}/out"; os.makedirs(OUT, exist_ok=True)
TOM = open(f"{OUT}/tom_tat.txt", "a")
def ghi(*a):
    s = f"[{(time.time()-T0)/3600:5.2f} h] " + " ".join(map(str, a)); print(s, flush=True); TOM.write(s + "\n"); TOM.flush()
mp = [m for m in glob.glob("/kaggle/input/**/harness/score_run.py", recursive=True) if "class PhiGround" in open(m, encoding="utf-8").read()]
assert mp, "không có score_run.py có PhiGround (Add Data phiground-ck)"
SRC = os.path.dirname(os.path.dirname(mp[0]))
if os.path.exists(f"{WS}/harness"): shutil.rmtree(f"{WS}/harness")
shutil.copytree(f"{SRC}/harness", f"{WS}/harness")
SR = f"{WS}/harness/score_run.py"
im = sorted(glob.glob("/kaggle/input/**/ep18171_s1.png", recursive=True)); assert im, "thiếu ảnh (Add Data thesis-score)"
TEST = f"{WS}/harness/dg1_cache/test_ac"
if not os.path.exists(f"{TEST}/images"): os.symlink(os.path.dirname(im[0]), f"{TEST}/images")
PR = os.path.dirname(sorted(glob.glob("/kaggle/input/**/preds/preds_kiem.jsonl", recursive=True))[0])
env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
ghi("harness", SR, "· ảnh", os.path.dirname(im[0]), "· preds", PR)
import torch, transformers
ghi("torch", torch.__version__, "· transformers", transformers.__version__, "· GPU", torch.cuda.device_count(), torch.cuda.get_device_name(0))
def doc(f):
    D = []
    if os.path.exists(f):
        for l in open(f, encoding="utf-8"):
            try: D.append(json.loads(l))
            except Exception: pass
    return D
def chay(ten, args, raw):
    log = open(f"{OUT}/{ten}.log", "a")
    p = subprocess.Popen([sys.executable, "-u", SR, "--grounder", "phiground", *args], stdout=log, stderr=subprocess.STDOUT, cwd=WS, env=env, start_new_session=True)
    while p.poll() is None:
        time.sleep(60)
        d = open(f"{OUT}/{ten}.log").read().strip().splitlines()[-1:] or [""]
        print(f"  {(time.time()-T0)/3600:5.2f} h · {ten}: {len(doc(raw))} · {d[0][:100]}", flush=True)
        if (time.time() - T0) / 3600 > TRAN_GIO: ghi("chạm trần giờ — dừng"); p.terminate(); time.sleep(20); break
    ghi(ten, "mã thoát", p.poll(), "·", len(doc(raw)), "bước")
    return p.poll()
# 24/10 chạy trọn Phi-Ground, tiền xử lý letterbox (chọn trên 100 bước câu chuẩn, xem report)
subprocess.run("pip install -q transformers==4.48.2 2>&1 | tail -3", shell=True)
ghi("transformers", subprocess.run([sys.executable, "-c", "import transformers;print(transformers.__version__)"], capture_output=True, text=True).stdout.strip())
env["PHIG_PRE"] = "letterbox"
for nh in ['ck500', 'base']:
    raw = f"{OUT}/score_{nh}_raw.jsonl"
    rc = chay(f"score_{nh}", ["--mode", "score", "--preds", f"{PR}/preds_{nh}.jsonl", "--out", f"{OUT}/score_{nh}.json"], raw)
    ghi("XONG", nh, "mã thoát", rc, "·", len(doc(raw)), "bước")
subprocess.run(f"cd {WS} && zip -qr phiground_bc.zip out", shell=True)
ghi("KẾT THÚC")